# Homework 03: Blending and multiperiod planning

**ISyE 524 — Fall 2026**  
**Due:** Monday, September 28, 2026, at 11:59 p.m. (Madison time).

Complete all three problems. For each problem, write a mathematical formulation before implementing it in Julia/JuMP. Define your variables, units, objective, constraints, and boundary conditions. Use continuous variables throughout this assignment.

The problems draw on Lectures 6–7: blending, inventory balances, staffing, and backlogging.


## Getting started and written answers

Follow [Assignments and PDF Submission](https://github.com/jlinderoth/isye524-students-julia/blob/main/docs/canvas-assignment-workflow.md). In VS Code, use **Tasks: Run Task** to run **ISyE 524: Update course repository**, then **ISyE 524: Start an assignment from its template**, entering `hw03`. Open `student-work/hw03/hw03.ipynb` and select the Julia 1.12 kernel. An existing working copy will not be overwritten. Later course updates refresh the template in `assignments/hw03/`, but do not automatically update your personal assignment copy.

For **every written part**, type LaTeX mathematics in Markdown or insert a clear photograph/scan of handwritten work. Include the answer and reasoning together and label the subparts; no duplicate transcription is required. Julia/JuMP implementations belong in code cells.

Use `$x_1 \geq 0$` for inline mathematics and `$$x_1+x_2\leq5$$` for display mathematics. Run the Markdown cell to render it.

For handwritten work, edit a Markdown cell in VS Code, drag in a cropped PNG/JPEG, and choose **Insert Image as Attachment**. Alternatively, put the image beside your working notebook under `student-work/hw03/images/` and insert `![My answer](images/problem1.png)`. Keep external images with the notebook. See [Images and handwritten work](https://github.com/jlinderoth/isye524-students-julia/blob/main/docs/assignments.md#images-and-handwritten-work).

Back up `student-work/hw03/` separately; Git does not back up your answers or images. After completing the notebook, export it with the course PDF task, inspect `submissions/hw03.pdf` for readable mathematics, tables, outputs, and images, and submit that PDF to Gradescope.


## Computational setup

Run the supplied setup and data cells in order. All data are included in this notebook. Use JuMP and HiGHS for every optimization model. Print the termination status and check that it is `MOI.OPTIMAL` and that a solution is available before reading values. Use unrounded values when checking constraints.

AI tools may help debug your own code, explain syntax or error messages, and critique your work. They should not replace constructing your formulation, implementation, or written solution. Complete the required collaboration and LLM-use statement at the end.


In [ ]:
using JuMP, HiGHS, Printf
import MathOptInterface as MOI

# Prefer PNG if you add plots to your answers.
if isdefined(Main, :IJulia)
    filter!(mime -> mime != MIME("image/svg+xml"), IJulia.ijulia_mime_types)
end


## 1: Alloy Blending

Prof. Linderoth is planning on quitting his job as a professor to become an oligarch that controls the steel industry. But before I can do that, I need your help to optimize a specific order for steel that my company LOSE (Linderoth Optimal Steel Emporium) recently received.

I need to deliver an order for 500 tons of steel for the Naval shipyards in Norfolk. The steel must be composed of the following grades:

| Element | Minimum (%) | Maximum (%) |
| :-- | --: | --: |
| Carbon (C) | 2 | 3 |
| Copper (Cu) | 0.4 | 0.6 |
| Manganese (Mn) | 1.2 | 1.65 |

LOSE has seven different types of raw material in stock to complete this order. They each have different chemical composition and also cost me a specified amount per ton. There is also a maximum amount of each type available. This data is given below:

| Raw | C (%) | Cu (%) | Mn (%) | Available (tons) | Cost (dollars/ton) |
| :-- | --: | --: | --: | --: | --: |
| F | 2.5 | 0 | 1.3 | 400 | 200 |
| U | 3 | 0.3 | 1.8 | 300 | 250 |
| E | 1.5 | 0.5 | 1.5 | 600 | 150 |
| L | 0 | 0.9 | 0 | 500 | 220 |
| O | 0.4 | 0.9 | 1.1 | 200 | 240 |
| N | 0 | 0.4 | 1.2 | 300 | 200 |
| M | 0 | 0.6 | 0 | 250 | 165 |

The order requires **at least 500 tons**. The customer permits excess delivery but pays no extra for it. All material blended is delivered. Quantities are continuous. Assume no material is lost in blending, and elemental composition is the mass-weighted average of the raw materials' compositions. Only the quantities actually used incur the listed costs.

### 1(a): Mathematical formulation

Write a linear programming instance that determines the number of tons of each raw that I need to put into my steel delivery to meet the grade requirements. My objective is to minimize the cost of the raw materials used to make the steel. Define variables and units; include the delivery requirement, raw-material availability limits, and both the lower and upper composition requirements for every element.

You may use indexed notation: $J$ for raw materials, $E$ for elements, $c_j$ for cost, $u_j$ for availability, $\gamma_{ej}$ for the percentage of element $e$ in raw $j$, $\alpha_e,\beta_e$ for minimum and maximum percentages, and $d=500$ for the minimum delivery.

Starting with a weighted-average expression for an element's percentage, explain why you can rewrite each quality requirement as a linear constraint. Identify why multiplication by the denominator is valid here. Use a consistent convention for percentages: for example, 2 and 3, or 0.02 and 0.03, throughout.


**Formulation and reasoning for 1(a):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work. Label the subparts.


### Supplied data for Problem 1

The following cell supplies the table values using the indexed notation above. In `γ[(e, j)]`, the element comes first and the raw material second.


In [ ]:
J = ["F", "U", "E", "L", "O", "N", "M"]
E = ["C", "Cu", "Mn"]
c = Dict(zip(J, [200, 250, 150, 220, 240, 200, 165]))
u = Dict(zip(J, [400, 300, 600, 500, 200, 300, 250]))
α = Dict(zip(E, [2.0, 0.4, 1.2]))
β = Dict(zip(E, [3.0, 0.6, 1.65]))
d = 500.0

# Rows follow J; columns follow E. Entries are percentages.
composition = [
    2.5  0.0  1.3
    3.0  0.3  1.8
    1.5  0.5  1.5
    0.0  0.9  0.0
    0.4  0.9  1.1
    0.0  0.4  1.2
    0.0  0.6  0.0
]
γ = Dict((e, j) => composition[k, l]
    for (k, j) in enumerate(J), (l, e) in enumerate(E))


### 1(b): Implement, solve, and check the blend

Implement your LP in JuMP and solve with HiGHS. Print the status, cost, tons of each raw used, and total tons delivered. Report the amount of each raw left unused.

Compute the percentage of C, Cu, and Mn in the resulting blend from the solved quantities. Compare each with both its minimum and maximum requirement. Which quality limits hold with equality? Check all availability limits and recompute the cost from the quantities.


In [ ]:
# Build and solve the blending LP here.
# Check mass, availability, each quality bound, and cost.


**Results and interpretation for 1(b):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work. Label the subparts.


### 1(c): At least 500 versus exactly 500

Without another solver run, show that **every** minimum-cost solution to this instance delivers exactly 500 tons.

If a feasible blend delivers $T>500$ tons, consider multiplying every quantity by $500/T$. Explain what happens to composition, availability, delivery, and cost. Which assumption about the costs makes the improvement strict?

Would replacing the delivery constraint by equality leave the entire feasible set unchanged, or only preserve the optimal decisions and value for this instance? Explain.


**Reasoning for 1(c):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work. Label the subparts.


## 2: So Much Grading!

The TAs and I have *so much grading* to do. Over the next five months, we anticipate that we will need additional staffing to complete our grading efforts. During each of the next five months it is estimated that the number of hours of grading that is necessary is given in the table below.

| Month | Grading (hours) |
| :-- | --: |
| September | 450 |
| October | 550 |
| November | 600 |
| December | 450 |
| January | 400 |

The UW pays 1,500 dollars per month per grader and each grader can handle as many as 16 hours of grading per month. We currently employ 30 graders. Graders can be hired only at the beginning of any month, and for each grader that is hired, the department incurs a one-time cost of 500 dollars. We can also ask a grader to retire (or we can fire them), for a one-time cost of 1,000 dollars per grader, at the beginning of a month.

Grading need not be completed in the month it arrives, i.e., some of it can be "carried over" to later months. However, it is bad to take too long for grading homeworks, so the department penalizes itself 1 dollar for each hour of ungraded homework at the end of the month. All grading must be done by the end of the fifth month.

Assume there is no unfinished grading before September. Work cannot be completed before it arrives. Graders hired at the beginning of a month can work that month, and graders who retire then receive no salary that month. Treat staffing as continuous grader equivalents for this LP; no rounding or integer restrictions are required. There is no prescribed final workforce size.

### 2(a): Mathematical formulation

Formulate a linear program whose solution will determine how many (if any) graders to hire at the beginning of each month and how many (if any) to ask to retire at the beginning of each month in order to minimize the total cost to the department over the next five months, including the penalty cost of not completing the work on time.

Define your variables and write the workforce and unfinished-work balances, capacity limits, initial conditions, and final requirement. You may use $T=\{1,\ldots,5\}$ with $W_t$ denoting the grading hours that arrive in month $t$.


**Formulation and reasoning for 2(a):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


### Supplied data for Problem 2

Use integer month indices in chronological order. Define your own decision-variable names in your formulation and implementation.


In [ ]:
T = 1:5
month_name = ["September", "October", "November", "December", "January"]
W = [450, 550, 600, 450, 400]
InitialGraders = 30
HoursPerGrader = 16
Salary = 1500
HireCost = 500
RetireCost = 1000
BacklogCost = 1


### 2(b): Implement, solve, and interpret

Implement and solve your model from part 2(a) using Julia and JuMP. Be sure to show the number of graders hired and retired each month, and the number of hours of homework left ungraded at the end of each month.

Also report the status, minimum total cost, workforce size, and hours completed each month. Check the monthly balances and capacity limits and confirm that all grading is finished by January. Explain how the plan trades staffing costs against delayed grading, and interpret any fractional staffing values under the continuous assumption.


In [ ]:
# Build and solve your staffing model here. Report and check the monthly plan.


**Results and interpretation for 2(b):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


## 3: The Goblet of Wine

In this problem, at Dumbledore's request, we will be producing goblets for drinking wine. There are six types of goblets $G=\{g_1,g_2,\ldots,g_6\}$ that are produced in batches of 1000 goblets. We will help Dumbledore plan his production for the next 12 weeks. Production can be incomplete—that is, a batch can contain fewer than 1000 goblets. The demand, in batches (or thousands), for each type of goblet over the next 12 weeks is given below.

| Goblet | Week 1 | 2 | 3 | 4 | 5 | 6 |
| :-- | --: | --: | --: | --: | --: | --: |
| $g_1$ | 20 | 22 | 18 | 35 | 17 | 19 |
| $g_2$ | 17 | 19 | 23 | 20 | 11 | 10 |
| $g_3$ | 18 | 35 | 17 | 10 | 9 | 21 |
| $g_4$ | 31 | 45 | 24 | 38 | 41 | 20 |
| $g_5$ | 23 | 20 | 23 | 15 | 10 | 22 |
| $g_6$ | 22 | 18 | 20 | 19 | 18 | 35 |

| Goblet | Week 7 | 8 | 9 | 10 | 11 | 12 |
| :-- | --: | --: | --: | --: | --: | --: |
| $g_1$ | 23 | 20 | 29 | 30 | 28 | 32 |
| $g_2$ | 12 | 34 | 21 | 23 | 30 | 12 |
| $g_3$ | 23 | 15 | 10 | 0 | 13 | 17 |
| $g_4$ | 19 | 37 | 28 | 12 | 30 | 37 |
| $g_5$ | 18 | 30 | 28 | 7 | 15 | 10 |
| $g_6$ | 0 | 28 | 12 | 30 | 21 | 23 |

For each type of goblet, the initial stock is known, as well as the required final stock level in thousands. The positive final stock level is in order for Dumbledore to combat the evil "end of horizon" effects. Per batch for every goblet type, the production and storage costs (in monetary units known as galleons) are also known. House elves make the goblets on magical machines, and the required working time and machine time (in hours) is also given. Storing the goblets takes space, and the required storage space per batch (measured in number of trays) is also given.

| Quantity | $g_1$ | $g_2$ | $g_3$ | $g_4$ | $g_5$ | $g_6$ |
| :-- | --: | --: | --: | --: | --: | --: |
| Production cost (galleons/batch) | 100 | 80 | 110 | 90 | 200 | 140 |
| Holding cost (galleons/batch/week) | 25 | 28 | 25 | 27 | 10 | 20 |
| Initial stock (batches) | 50 | 20 | 0 | 15 | 0 | 10 |
| Minimum final stock (batches) | 10 | 10 | 10 | 10 | 10 | 10 |
| Elf time (hours/batch) | 3 | 3 | 3 | 2 | 4 | 4 |
| Machine time (hours/batch) | 2 | 1 | 4 | 8 | 11 | 9 |
| Storage space (trays/batch) | 4 | 5 | 5 | 6 | 4 | 9 |

There is a limited amount of elf time and machine time available per week. Specifically, the number of working hours is limited to 420 hours/week, and the machines have a weekly capacity of 800 hours. In Hogwarts, storage space for at most 1000 trays is available.

Initially, all demand must be met in the week it occurs. Charge holding costs on stock at the end of each week, including week 12, and apply the storage limit to these same end-of-week stocks. Final stock requirements are lower bounds. Fractional batches are allowed.

### 3(a): Mathematical formulation

Formulate a linear program to determine the different types of goblets that need to be produced in each period to minimize the total cost of production and storage. Use a general indexed formulation and define all variables and parameters. Include initial and final stock conditions and each shared resource limit.


**Formulation and reasoning for 3(a):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


### Supplied data for Problem 3

Rows of `demand` follow `G`; columns are weeks 1–12. The dictionary `d[(g, t)]` gives the demand for goblet type `g` in week `t`. The code uses `I0`, `IFinal`, and `machine_time` for the initial stock, final stock, and machine hours per batch. `c`, `h`, `w`, and `s` give production cost, holding cost, elf hours, and storage trays per batch, respectively.


In [ ]:
G = ["g1", "g2", "g3", "g4", "g5", "g6"]
T = 1:12
demand = [
    20 22 18 35 17 19 23 20 29 30 28 32
    17 19 23 20 11 10 12 34 21 23 30 12
    18 35 17 10  9 21 23 15 10  0 13 17
    31 45 24 38 41 20 19 37 28 12 30 37
    23 20 23 15 10 22 18 30 28  7 15 10
    22 18 20 19 18 35  0 28 12 30 21 23
]
d = Dict((g, t) => demand[i, t] for (i, g) in enumerate(G), t in T)
c = Dict(zip(G, [100, 80, 110, 90, 200, 140]))
h = Dict(zip(G, [25, 28, 25, 27, 10, 20]))
I0 = Dict(zip(G, [50, 20, 0, 15, 0, 10]))
IFinal = Dict(g => 10 for g in G)
w = Dict(zip(G, [3, 3, 3, 2, 4, 4]))
machine_time = Dict(zip(G, [2, 1, 4, 8, 11, 9]))
s = Dict(zip(G, [4, 5, 5, 6, 4, 9]))
H = 420
M = 800
S = 1000


### 3(b): Implement and solve

Implement your model from part 3(a) in Julia/JuMP and solve it with HiGHS. Display the status and minimum cost. Report the optimal production quantities and end-of-week inventory for each goblet type, with a separate summary of the end-of-week inventory for type $g_2$.

Check the stock balances, weekly elf and machine capacities, storage limits, and final stock requirements. Briefly explain why some goblets may need to be produced before the week in which they are demanded.


In [ ]:
# Build and solve the goblet production model. Report and check the plan.


**Results and interpretation for 3(b):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


### 3(c): Allow backlogging

Determine how the production schedule changes if goblet demand is allowed to be backlogged at **zero** cost. The final inventory requirements must still be met, and all demand must have been filled by the end of week 12. All other data remain as in parts 3(a)–3(b). Holding costs and tray usage apply only to physical stock on hand.

Write the changes to your mathematical model, defining any new variables. Distinguish physical stock, backlog, and net inventory. Implement the modified model and report the status, minimum cost, and end-of-week **net inventory** for $g_2$ in every week. Compare with part 3(b), identifying when $g_2$ is backlogged and the cost saving. Check that every backlog is zero and every final stock requirement is met at the end of week 12.

Explain why allowing delayed demand cannot increase the minimum cost.


**Formulation changes and reasoning for 3(c):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


In [ ]:
# Implement the backlog model here and compare its results with part 3(b).


**Results and interpretation for 3(c):**

Type your answer and reasoning in Markdown, or insert a clear photograph/scan of handwritten work.


## Collaboration and LLM-use statement — required

Complete this statement as the **very last part of your homework**. Be specific enough that a reader can understand what work you did yourself and how others or LLMs helped you learn.

- **People:** Give the name of every person you collaborated with, identify the problem/subpart you discussed, and explain the nature of the collaboration. If you worked alone, explicitly state that you did not collaborate with anyone.
- **LLMs/tools:** Name every LLM or AI assistant you used and identify the model/version as precisely as available. If the interface did not show a model/version, say so. If you used none, explicitly state that you did not use an LLM or AI assistant.
- **Specific assistance and learning:** For each tool, identify the problem/subpart, describe what you asked it to help with, what suggestions you used or rejected, and how the interaction helped you understand the mathematics or implementation. Explain how you checked the suggestions. “I used ChatGPT for help” is not sufficiently specific.

Disclosure does not replace the requirement that your submitted formulation, implementation, and explanation be your own work and that you understand them.


**People and collaboration:**

[Names, problem/subparts, and what you discussed; or explicitly state that you worked alone.]

**LLMs/tools and models:**

[Tool names and models/versions, or explicitly state that you used none.]

**How I used the assistance, what I learned, and how I checked it:**

[Specific details for each tool and relevant problem/subpart; state not applicable if you used none.]
